In [2]:
!pip install -q transformers datasets accelerate bitsandbytes scipy --break-system-packages

In [3]:
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os, gc, re, time, random
import torch
from collections import Counter
from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata

In [4]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Successfully logged into Hugging Face.")

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [6]:
# Configs
MODEL_ID   = "meta-llama/Llama-3.2-3B-Instruct"
QUANT_TYPE = "W4A16" # or W8A16
SELF_CONSISTENCY_N    = 5
SELF_CONSISTENCY_TEMP = 0.7
MAX_NEW_TOKENS        = 1024
SEED                  = 42
OUTPUT_DIR            = "/content/drive/MyDrive/quantization_research"
OUTPUT_PATH           = os.path.join(OUTPUT_DIR, "llama_gsm8k_w4a16.csv")


# For W8A16:
# bnb_config = BitsAndBytesConfig(load_in_8bit=True)

# For W4A16:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [7]:
# Define helper functions
def count_steps(s): return len(re.findall(r"<<", s))

def gt_from(ans):
    m = re.search(r"####\s*(.+)", ans)
    return m.group(1).strip().replace(",", "") if m else None

def build_prompt(q):
    return (
        "Solve this math problem step by step. "
        "Show each step clearly numbered (Step 1, Step 2, ...). "
        "End with 'Final Answer: <number>'.\n\n"
        f"Problem: {q}\n\nSolution:"
    )

def generate_text(prompt, temperature=0.0, max_tokens=MAX_NEW_TOKENS):
    msgs = [{"role": "user", "content": prompt}]
    txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(txt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        if temperature == 0:
            out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        else:
            out = model.generate(**inputs, max_new_tokens=max_tokens,
                                 do_sample=True, temperature=temperature, top_p=0.95)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True), len(gen)

def extract_answer(text):
    m = re.search(r"[Ff]inal\s*[Aa]nswer\s*:\s*\$?\s*([\-\d,\.]+)", text)
    if m: return m.group(1).replace(",", "").strip()
    m = re.search(r"####\s*([\-\d,\.]+)", text)
    if m: return m.group(1).replace(",", "").strip()
    nums = re.findall(r"([\-\d,]+\.?\d*)", text)
    return nums[-1].replace(",", "").strip() if nums else None

def is_correct(pred, gt):
    if pred is None or gt is None: return False
    try: return abs(float(pred) - float(gt)) < 1e-2
    except ValueError: return str(pred).strip() == str(gt).strip()

def first_error_step(text):
    labeled = re.findall(
        r"[Ss]tep\s+(\d+)[^\n]{0,80}?(\d+\.?\d*)\s*([\+\-\*x×/÷])\s*(\d+\.?\d*)\s*=\s*(\d+\.?\d*)",
        text)
    if labeled:
        candidates = labeled
    else:
        raw = re.findall(r"(\d+\.?\d*)\s*([\+\-\*x×/÷])\s*(\d+\.?\d*)\s*=\s*(\d+\.?\d*)", text)
        candidates = [(str(i+1), a, op, b, r) for i, (a, op, b, r) in enumerate(raw)]
    for step_num, a, op, b, result in candidates:
        try:
            a_f, b_f, r_f = float(a), float(b), float(result)
            if op == "+": expected = a_f + b_f
            elif op == "-": expected = a_f - b_f
            elif op in ["*","x","×","·"]: expected = a_f * b_f
            elif op in ["/","÷"]: expected = a_f / b_f if b_f != 0 else float("inf")
            else: continue
            tol = max(0.01, abs(expected) * 0.01)
            if abs(expected - r_f) > tol:
                return int(step_num)
        except (ValueError, ZeroDivisionError):
            continue
    return None

def majority_vote(texts):
    answers = [extract_answer(t) for t in texts]
    valid = [a for a in answers if a is not None]
    if not valid: return None, texts[0] if texts else ""
    winner = Counter(valid).most_common(1)[0][0]
    for t, a in zip(texts, answers):
        if a == winner: return winner, t
    return winner, texts[0]

def save_row(row):
    pd.DataFrame([row]).to_csv(
        OUTPUT_PATH, mode="a",
        header=not os.path.exists(OUTPUT_PATH) or os.path.getsize(OUTPUT_PATH) == 0,
        index=False,
    )

In [8]:
random.seed(SEED) # Same seed = same set of problems sampled
torch.manual_seed(SEED)

# Load dataset
print("Loading GSM8K dataset...")
ds = load_dataset("openai/gsm8k", "main", split="test")


by_diff = {"easy": [], "medium": [], "hard": []}
for i, row in enumerate(ds):
    n = count_steps(row["answer"])
    if n <= 2: by_diff["easy"].append(i)
    elif n <= 4: by_diff["medium"].append(i)
    else: by_diff["hard"].append(i)

random.shuffle(by_diff["easy"])
random.shuffle(by_diff["medium"])
random.shuffle(by_diff["hard"])

sampled = by_diff["easy"][:20] + by_diff["medium"][:20] + by_diff["hard"][:10]

problems = []
for idx in sampled:
    row = ds[idx]
    n = count_steps(row["answer"])
    diff = "easy" if n <= 2 else ("medium" if n <= 4 else "hard")
    problems.append({
        "problem_id":   idx,
        "question":     row["question"],
        "ground_truth": gt_from(row["answer"]),
        "difficulty":   diff,
        "ref_steps":    n,
    })
print("Sampled 50 problems: 20 easy, 20 medium, 10 hard")

Loading GSM8K dataset...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Sampled 50 problems: 20 easy, 20 medium, 10 hard


In [9]:
# Resume
if os.path.exists(OUTPUT_PATH):
    existing = pd.read_csv(OUTPUT_PATH)
    done_keys = set(zip(existing["decoding_strategy"], existing["problem_id"]))
    print(f"Existing CSV has {len(existing)} rows - skipping already-done.")
else:
    done_keys = set()
    print("No existing CSV - starting fresh.")

# Load model
print(f"\nLoading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

No existing CSV - starting fresh.

Loading meta-llama/Llama-3.2-3B-Instruct...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded. GPU memory: 2.38 GB


In [10]:
start = time.time()

for STRATEGY in ["greedy", "self_consistency"]:
    print(f"\n{'='*55}\nStrategy: {STRATEGY} | Quant: {QUANT_TYPE}\n{'='*55}")
    count = skipped = 0
    total = len(problems)

    for prob in problems:
        if (STRATEGY, prob["problem_id"]) in done_keys:
            skipped += 1
            continue
        count += 1
        prompt = build_prompt(prob["question"])

        if STRATEGY == "greedy":
            text, n_tokens = generate_text(prompt, temperature=0.0)
            pred = extract_answer(text)
            err_step = first_error_step(text)
            raw = text
            n_samples = 1
        else:
            texts, tok_counts = [], []
            for _ in range(SELF_CONSISTENCY_N):
                t, n = generate_text(prompt, temperature=SELF_CONSISTENCY_TEMP)
                texts.append(t); tok_counts.append(n)
            pred, best = majority_vote(texts)
            n_tokens = int(sum(tok_counts) / len(tok_counts))
            err_step = first_error_step(best)
            raw = best
            n_samples = SELF_CONSISTENCY_N

        correct = is_correct(pred, prob["ground_truth"])
        row = {
            "quant_type":              QUANT_TYPE,
            "decoding_strategy":       STRATEGY,
            "problem_id":              prob["problem_id"],
            "difficulty":              prob["difficulty"],
            "predicted_answer":        pred,
            "ground_truth":            prob["ground_truth"],
            "correct":                 correct,
            "reasoning_length_tokens": n_tokens,
            "first_error_step":        err_step,
            "raw_output":              raw,
            "n_samples":               n_samples,
        }
        save_row(row)
        done_keys.add((STRATEGY, prob["problem_id"]))

        elapsed = time.time() - start
        print(f"  [{count}/{total - skipped}] pid={prob['problem_id']:<5d} "
              f"diff={prob['difficulty']:<7s} pred={str(pred):<10s} "
              f"gt={prob['ground_truth']:<10s} correct={correct}  "
              f"err_step={err_step}  elapsed={elapsed/60:.0f}min")

# Summary
df = pd.read_csv(OUTPUT_PATH)
print(f"\n✅ DONE — {len(df)} rows in {OUTPUT_PATH}")
print(f"Total time: {(time.time()-start)/60:.1f} min\n")
print("="*55 + f"\nSUMMARY — {QUANT_TYPE}\n" + "="*55)
for strat in ["greedy", "self_consistency"]:
    sub = df[df["decoding_strategy"] == strat]
    if len(sub) == 0: continue
    acc = sub["correct"].mean() * 100
    print(f"  {strat}: {acc:.1f}% ({sub['correct'].sum()}/{len(sub)})")
    for diff in ["easy", "medium", "hard"]:
        d = sub[sub["difficulty"] == diff]
        if len(d) > 0:
            print(f"    {diff}: {d['correct'].mean()*100:.1f}% ({d['correct'].sum()}/{len(d)})")

del model, tokenizer
gc.collect(); torch.cuda.empty_cache()
print("\n✅ GPU memory cleared")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Strategy: greedy | Quant: W4A16


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [1/50] pid=473   diff=easy    pred=9.09       gt=10         correct=False  err_step=None  elapsed=1min


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

  [48/50] pid=465   diff=hard    pred=30         gt=30         correct=True  err_step=None  elapsed=80min


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [49/50] pid=199   diff=hard    pred=7500       gt=7500       correct=True  err_step=None  elapsed=82min


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [50/50] pid=830   diff=hard    pred=1128       gt=1128       correct=True  err_step=None  elapsed=84min

✅ DONE — 100 rows in /content/drive/MyDrive/quantization_research/llama_gsm8k_w4a16.csv
Total time: 83.9 min

SUMMARY — W4A16
  greedy: 66.0% (33/50)
    easy: 70.0% (14/20)
    medium: 70.0% (14/20)
    hard: 50.0% (5/10)
  self_consistency: 84.0% (42/50)
    easy: 80.0% (16/20)
    medium: 85.0% (17/20)
    hard: 90.0% (9/10)

✅ GPU memory cleared
